# 第04回：アテンションの計算とESM-2の穴埋め予測
**20分／採点・記述提出・回答送信なし**

このColabの中で、Pythonの計算と、表・図・アニメーションの表示を連動させる。
1. **アテンションの模型**：短い文章で、Query・Keyから重みを作り、Valueを混ぜる過程を見る。
2. **ESM-2の穴埋め予測**：ユビキチン（PDB 1UBQ、76残基）の一文字を隠し、候補を予測する。
3. **周囲の文脈の比較**：対象残基と組成を保ち、周囲の順序を変えると予測がどう変わるかを見る。

**開始：メニュー「ランタイム → すべてのセルを実行」**。準備後は2つの操作画面が表示される。選択や数値を変えるだけでPythonが再計算するので、条件ごとにセルを実行し直す必要はない。

無料アカウントのCPUランタイムで利用できる。GPU・APIキー・Driveのマウントは不要。モデルを取得できなかった場合は、同じ操作画面で「保存済みの実計算」を表示する。今回の推論か保存済み結果かは画面に明示する。

表示欄が空白のままなら、その操作画面のセルを再実行する。直らない場合はページを再読み込みして全セルを実行し、接続完了後に表示セルを再実行する。編集内容を残すときは「ドライブにコピーを保存」を使う。


In [ ]:
# @title 1. 操作画面を準備する
%pip -q install anywidget==0.9.21
from google.colab import output
output.enable_custom_widget_manager()
import importlib.metadata
print('表示の準備完了：anywidget', importlib.metadata.version('anywidget'))


## 1　どの情報を、どれだけ混ぜるか
最初の「猫は庭で魚を食べる」は、手で設定したQ・K・Vによる1ヘッドの説明用模型である。語の意味を学習したモデルではない。講義の「3位置の数値例」へ切り替えることもできる。

- 「食べる」の行を選び、表のセルを押してQueryとKeyの内積を読む。
- 「① QとKのスコア → ② softmaxで重みへ → ③ Valueを混ぜる」と進み、「混ぜる過程を再生」を押す。
- Queryを「魚」にして因果マスクをオンにすると、未来の「を」「食べる」の重みが0になる。
- 温度を下げると重みの差が強調される。Valueだけを変えると、重みは同じでも出力が変わる。

因果マスクは未来の参照を止める操作であり、ESM-2で文字を隠す［MASK］とは異なる。


### 計算の流れ
入力の表現からQ・K・Vを作り、次を計算します。この例では $H=I$ とおけば、設定したQ・K・Vをそれぞれ $W_Q,W_K,W_V$ とみなせます。ベクトルの各成分に、語の意味の名前を付けているわけではありません。

$$s_{ij}=\frac{Q_i\cdot K_j}{\sqrt{d_k}},\qquad a_{ij}=\frac{\exp(s_{ij}/T)}{\sum_k\exp(s_{ik}/T)},\qquad O_i=\sum_j a_{ij}V_j.$$

この試作では $d_k=2$、標準の温度は $T=1$ です。因果マスクを使う場合は、未来の位置のlogitを $-\infty$ にしてからsoftmaxを計算します。

講義の3位置の例では、位置1・マスクなし・T=1で、重みは約(0.4011, 0.1978, 0.4011)、出力は約(1.2033, 0.7967)となる。


In [ ]:
# @title 2. アテンションの計算を定義する
"""教育用セルフアテンション。計算はすべてPythonで行う。"""
import math
import numpy as np


def preset(name="sentence"):
    if name == "lecture":
        return ["位置1", "位置2", "位置3"], np.array([[1., 0.], [0., 1.], [1., 1.]]), np.array([[1., 0.], [0., 1.], [1., 1.]]), np.array([[1., 0.], [0., 2.], [2., 1.]])
    if name != "sentence":
        raise ValueError("未知の例です。")
    # 語の意味を学習した数値ではない。H=I とすると各行列をW_Q, W_K, W_Vとみなせる。
    tokens = ["猫", "は", "庭", "で", "魚", "を", "食べる"]
    q = np.array([[1., .2], [.2, .8], [.2, 1.2], [.2, .8], [1.2, .2], [.2, .8], [1.4, .2]])
    k = np.array([[1.6, .1], [.1, .4], [.2, 1.6], [.1, .4], [1.4, .2], [.1, .4], [.4, .8]])
    v = np.array([[1., 0.], [.1, .2], [0., 1.], [.2, .1], [1.6, .4], [.1, .2], [.6, 1.2]])
    return tokens, q, k, v


def calculate(request=None):
    r = request or {}
    name = r.get("preset", "sentence")
    tokens, q, k, v = preset(name)
    n = len(tokens)
    i = int(r.get("query", n - 1 if name == "sentence" else 0))
    j = int(r.get("key", 0))
    temperature = float(r.get("temperature", 1.))
    if not (0 <= i < n and 0 <= j < n and .25 <= temperature <= 2.):
        raise ValueError("位置または温度の範囲が不正です。")
    if "q" in r:
        new_q = np.asarray(r["q"], dtype=float)
        if new_q.shape != (2,) or not np.isfinite(new_q).all() or (np.abs(new_q) > 3).any():
            raise ValueError("Queryは−3〜3の2成分で指定します。")
        q[i] = new_q
    if "v" in r:
        new_v = np.asarray(r["v"], dtype=float)
        if new_v.shape != (n, 2) or not np.isfinite(new_v).all() or (np.abs(new_v) > 3).any():
            raise ValueError("Valueの形または値が不正です。")
        v = new_v
    dot = q @ k.T
    scores = dot / math.sqrt(k.shape[1])
    logits = scores / temperature
    mask = np.triu(np.ones((n, n), dtype=bool), 1) if r.get("causal", False) else np.zeros((n, n), dtype=bool)
    logits = np.where(mask, -np.inf, logits)
    exp_shifted = np.exp(logits - logits.max(axis=1, keepdims=True))
    weights = exp_shifted / exp_shifted.sum(axis=1, keepdims=True)
    contributions = weights[i, :, None] * v
    out = weights @ v
    # JSONにInfinity/NaNを入れず、マスク部分をnullで表す。
    return dict(preset=name, tokens=tokens, query=i, key=j, temperature=temperature,
                causal=bool(r.get("causal", False)), q=q.tolist(), k=k.tolist(), v=v.tolist(),
                dot=dot.tolist(), scores=scores.tolist(),
                logits=[[None if mask[a,b] else float(logits[a,b]) for b in range(n)] for a in range(n)],
                weights=weights.tolist(), contributions=contributions.tolist(),
                partials=np.cumsum(contributions, axis=0).tolist(), output=out.tolist(),
                row_sum=float(weights[i].sum()), request_id=r.get("request_id", 0))


In [ ]:
# @title 3. アテンションを操作する
# @title 3. インタラクティブ教材を開く
FRONTEND = 'function render({model, el}) {\n  const root = document.createElement(\'div\'); el.append(root);\n  const shadow = root.attachShadow({mode:\'open\'});\n  shadow.innerHTML = `<style>\n  :host{display:block;color:#172b42;background:#fff;font:15px/1.55 system-ui,-apple-system,\'Noto Sans JP\',sans-serif}\n  *{box-sizing:border-box}h2{font-size:25px;margin:0;color:#103d79}h3{font-size:18px;margin:14px 0 7px}p{margin:5px 0 10px}.app{max-width:1100px;padding:20px;margin:auto}\n  .note{color:#465c6e;font-size:13px}.controls,.steps,.tokens{display:flex;flex-wrap:wrap;align-items:center;gap:9px;margin:12px 0}\n  button,select,input{font:inherit}button,select{border:1px solid #aab9cb;border-radius:6px;background:white;color:#172b42;padding:7px 11px}\n  button{cursor:pointer}button:hover{background:#edf4ff}button[aria-pressed=true]{background:#174d91;color:white;border-color:#174d91}button:focus-visible,select:focus-visible,input:focus-visible{outline:3px solid #d26713;outline-offset:2px}\n  button:disabled{opacity:.5;cursor:wait}label{display:flex;align-items:center;gap:6px}input[type=number]{width:76px;border:1px solid #aab9cb;padding:5px;border-radius:4px}\n  .line{border-top:1px solid #d3dee9;margin-top:16px;padding-top:12px}.grid{display:grid;grid-template-columns:minmax(330px,1.2fr) minmax(270px,1fr);gap:22px}.scroll{overflow:auto}\n  table{border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums}th,td{text-align:center;padding:5px;border-bottom:1px solid #e1e7ef;font-size:13px}td button{padding:7px 2px;border:1px solid transparent;border-radius:3px;width:100%;font-size:13px}\n  .row-selected th{color:#174d91;border-left:4px solid #174d91}td button.selected{outline:3px solid #dd791c;outline-offset:-3px}th small{font-size:10px;font-weight:normal;display:block}\n  .formula{background:#f0f5fa;padding:12px;border-left:4px solid #174d91;font-variant-numeric:tabular-nums;overflow-wrap:anywhere;min-height:115px}.big{font-size:19px;font-weight:650}.status{font-size:12px;color:#466176;margin:8px 0}\n  svg{width:100%;height:auto;display:block}svg text{font-family:inherit}.flow{min-width:630px}.explanation{min-height:48px}.steps button{flex:1;min-width:145px}.stage-label{font-weight:650;color:#174d91}.editing{background:#fbf5ec;padding:10px;margin-top:8px}\n  @media(max-width:700px){.app{padding:10px}.grid{grid-template-columns:1fr;gap:8px}h2{font-size:21px}.controls{align-items:flex-start}.controls label{flex-wrap:wrap}}\n  @media(prefers-reduced-motion:reduce){*{animation:none!important;transition:none!important}}\n  </style><div class="app">\n  <h2>アテンション：どの情報を、どれだけ混ぜるか</h2>\n  <p class="note">説明用に設定したQ・K・Vを使う、1ヘッドのセルフアテンションです。学習済み言語モデルの重みではありません。</p>\n  <div class="controls"><label>例 <select aria-label="例"><option value="sentence">猫は庭で魚を食べる</option><option value="lecture">講義の3位置の数値例</option></select></label><label><input type="checkbox" aria-label="因果マスク">未来の位置を参照しない</label><label>温度 T <input aria-label="温度 T" type="number" min="0.25" max="2" step="0.25" value="1"></label><button data-action="reset">初期状態に戻す</button></div>\n  <p class="note">T=1が通常の式です。Tを下げると重みの違いが強調されます。因果マスクは未来の参照を止める操作で、ESM-2の文字を隠す［MASK］とは異なります。</p>\n  <h3>1\u3000更新する位置（Query）を選ぶ</h3><div class="tokens"></div>\n  <div class="controls qedit"><label>Qの第1成分 <input type="number" aria-label="Query 第1成分" min="-3" max="3" step="0.2"></label><label>Qの第2成分 <input type="number" aria-label="Query 第2成分" min="-3" max="3" step="0.2"></label><button data-action="calculate">Pythonで再計算</button></div>\n  <div class="status" role="status" aria-live="polite"></div>\n  <div class="steps"><button data-stage="0">① QとKのスコア</button><button data-stage="1">② softmaxで重みへ</button><button data-stage="2">③ Valueを混ぜる</button></div>\n  <p class="explanation"></p>\n  <div class="grid"><section><h3 class="matrix-title"></h3><p class="note">行＝更新するQueryの位置、列＝参照するKeyの位置。セルを押すと、そのペアを詳しく表示します。</p><div class="scroll matrix"></div></section>\n  <section><h3>選んだペアの計算</h3><div class="formula"></div><h3>各位置の入力ベクトル</h3><div class="scroll vectors"></div><div class="editing"><label>選択した位置のV 第1成分 <input type="number" aria-label="Value 第1成分" min="-3" max="3" step="0.2"></label><p class="note">Valueだけを変えると、重みは同じでも出力が変わります。</p></div></section></div>\n  <section class="line"><h3>参照先のValue → 選んだ位置の出力</h3><p class="note">線の太さは重み。動く点は情報を混ぜる順序の説明で、実装の実行順や物理的な移動ではありません。</p><div class="scroll"><div class="flow"></div></div>\n  <div class="controls"><button data-action="play">混ぜる過程を再生</button><button data-action="step">1つずつ加える</button><button data-action="stop">停止</button></div><div class="sum formula"></div></section>\n  <p class="line note">Q（Query）とK（Key）の内積から参照の重みを作り、V（Value）をその重みで足し合わせます。これは1ヘッドの出力で、Transformer全体の出力ではありません。重みだけから単語の意味や予測の理由が確定するわけではありません。</p>\n  </div>`;\n  const $ = s => shadow.querySelector(s), $$ = s => [...shadow.querySelectorAll(s)];\n  const fmt = x => x === null ? \'−∞\' : Number(x).toFixed(3);\n  const vec = x => `(${x.map(fmt).join(\', \')})`;\n  let data=null, stage=1, step=-1, timer=null, timeout=null, busy=false, pending=0;\n  function stop(){clearInterval(timer);timer=null;}\n  function params(){return {preset:data.preset,query:data.query,key:data.key,q:data.q[data.query],v:data.v,temperature:data.temperature,causal:data.causal};}\n  function send(overrides={}) {\n    if(busy)return; stop();step=-1;busy=true;\n    pending=Math.max(pending,model.get(\'request\')?.request_id||0)+1;\n    const req={...params(),...overrides,request_id:pending};\n    Object.keys(req).forEach(k=>{if(req[k]===undefined)delete req[k];});\n    $$(\'.app button,.app input,.app select\').forEach(e=>e.disabled=true);\n    $(\'.status\').textContent=\'ColabのPythonで計算中…\';\n    model.set(\'request\',req);model.save_changes();\n    clearTimeout(timeout); timeout=setTimeout(()=>{busy=false;$$(\'.app button,.app input,.app select\').forEach(e=>e.disabled=false);$(\'.status\').textContent=\'Pythonからの応答がありません。Colabの接続を確認し、「Pythonで再計算」を押してください。\';},15000);\n  }\n  function showFlow() {\n    if(!data)return; const n=data.tokens.length, i=data.query, a=data.weights[i];\n    const width=960, center=480, xs=data.tokens.map((_,j)=>65+j*830/(n-1));\n    const lines=a.map((w,j)=>`<path d="M${xs[j]} 87 Q${xs[j]} 170 ${center} 235" fill="none" stroke="${j===step?\'#d76c12\':\'#366dac\'}" stroke-width="${w===0?1:1+14*w}" opacity="${w===0?.13:.65}" ${w===0?\'stroke-dasharray="4 5"\':\'\'}/>${j===step&&w>0?`<circle r="7" fill="#d76c12"><animateMotion dur="0.8s" fill="freeze" path="M${xs[j]} 87 Q${xs[j]} 170 ${center} 235"/></circle>`:\'\'}`).join(\'\');\n    const boxes=data.tokens.map((t,j)=>`<g><rect x="${xs[j]-58}" y="14" width="116" height="73" rx="7" fill="${j===step?\'#fff0df\':\'#f0f5fa\'}" stroke="${j===step?\'#d76c12\':\'#b7c9dc\'}"/><text x="${xs[j]}" y="36" text-anchor="middle" font-size="17">${j+1} ${t}</text><text x="${xs[j]}" y="58" text-anchor="middle" font-size="12">V=${vec(data.v[j])}</text><text x="${xs[j]}" y="77" text-anchor="middle" font-size="13">重み ${fmt(a[j])}</text></g>`).join(\'\');\n    const partial=step>=0?data.partials[step]:data.output[i];\n    $(\'.flow\').innerHTML=`<svg role="img" aria-label="ValueからQueryへの加重和。線の太さが重み。" viewBox="0 0 960 310">${lines}${boxes}<rect x="310" y="235" width="340" height="65" rx="8" fill="#174d91"/><text x="480" y="259" text-anchor="middle" fill="white" font-size="16">${step>=0?\'ここまでの和\':\'出力\'}：位置${i+1} ${data.tokens[i]}</text><text x="480" y="284" text-anchor="middle" fill="white" font-size="20">${vec(partial)}</text></svg>`;\n    $(\'.sum\').innerHTML=step<0?`<span class="big">出力 O${i+1} = Σⱼ a${i+1},ⱼ Vⱼ = ${vec(data.output[i])}</span><br>「1つずつ加える」で、各Valueがどれだけ加わるかを見ます。`:`位置${step+1}「${data.tokens[step]}」：${fmt(a[step])} × ${vec(data.v[step])} = ${vec(data.contributions[step])}<br><span class="big">ここまでの和 = ${vec(partial)}</span>${step===n-1?\'<br>すべて加えると出力と一致します。\':\'\'}`;\n  }\n  function draw() {\n    if(!data)return;\n    const {tokens,q,k,v,query:i,key:j}=data, n=tokens.length;\n    $(\'select\').value=data.preset; $(\'[aria-label="因果マスク"]\').checked=data.causal; $(\'[aria-label="温度 T"]\').value=data.temperature;\n    $(\'[aria-label="Query 第1成分"]\').value=q[i][0];$(\'[aria-label="Query 第2成分"]\').value=q[i][1];$(\'[aria-label="Value 第1成分"]\').value=v[j][0];\n    $(\'.tokens\').innerHTML=tokens.map((t,z)=>`<button aria-label="Query ${z+1} ${t}" data-query="${z}" aria-pressed="${z===i}">${z+1} ${t}</button>`).join(\'\');\n    $$(\'.tokens button\').forEach(b=>b.onclick=()=>{const z=+b.dataset.query;send({query:z,q:data.q[z]});});\n    $$(\'[data-stage]\').forEach(b=>b.setAttribute(\'aria-pressed\',+b.dataset.stage===stage));\n    $(\'.explanation\').innerHTML=[`<span class="stage-label">① スコアを作る：</span>sᵢⱼ = (Qᵢ · Kⱼ) / √2。Q・Kは2成分です。選択した行が、更新する位置の参照スコアです。`,`<span class="stage-label">② 重みを作る：</span>aᵢⱼ = exp(sᵢⱼ / T) / Σₖ exp(sᵢₖ / T)。行ごとに正規化するので、各行の和は1です。`,`<span class="stage-label">③ 情報を混ぜる：</span>Oᵢ = Σⱼ aᵢⱼ Vⱼ。下の図でValueを順に足します。出力の各成分は、同じ重みを使った平均です。`][stage];\n    $(\'.matrix-title\').textContent=stage===0?\'全ペアのスコア sᵢⱼ\':\'全ペアのアテンション重み aᵢⱼ\';\n    const matrix=stage===0?data.scores:data.weights;\n    $(\'.matrix\').innerHTML=`<table><thead><tr><th>Q ↓ / K →</th>${tokens.map((t,z)=>`<th>${t}<small>${z+1}</small></th>`).join(\'\')}</tr></thead><tbody>${matrix.map((row,r)=>`<tr class="${r===i?\'row-selected\':\'\'}"><th>${tokens[r]}<small>${r+1}</small></th>${row.map((x,c)=>`<td><button data-r="${r}" data-c="${c}" aria-label="ペア ${r+1} ${c+1}" class="${r===i&&c===j?\'selected\':\'\'}" style="background:rgba(35,106,180,${.06+.55*data.weights[r][c]})">${fmt(x)}</button></td>`).join(\'\')}</tr>`).join(\'\')}</tbody></table>`;\n    $$(\'.matrix button\').forEach(b=>b.onclick=()=>send({query:+b.dataset.r,key:+b.dataset.c,q:q[+b.dataset.r]}));\n    const masked=data.logits[i][j]===null;\n    $(\'.formula\').innerHTML=`<b>Query：${i+1} ${tokens[i]} → Key：${j+1} ${tokens[j]}</b><br>Q = ${vec(q[i])} ／ K = ${vec(k[j])}<br>内積 = ${fmt(q[i][0])} × ${fmt(k[j][0])} + ${fmt(q[i][1])} × ${fmt(k[j][1])} = ${fmt(data.dot[i][j])}<br>スコア = ${fmt(data.dot[i][j])} / √2 = ${fmt(data.scores[i][j])}<br>${masked?\'因果マスク：softmaxに入れる値を−∞にして重みを0にする。\':`softmaxへ入れる値 = スコア / T = ${fmt(data.logits[i][j])}`}<br><span class="big">重み a = ${fmt(data.weights[i][j])}</span> ／ この行の和 = ${data.row_sum.toFixed(6)}`;\n    $(\'.vectors\').innerHTML=`<table><tr><th>位置</th><th>Q</th><th>K</th><th>V</th></tr>${tokens.map((t,z)=>`<tr><th>${z+1} ${t}</th><td>${vec(q[z])}</td><td>${vec(k[z])}</td><td>${vec(v[z])}</td></tr>`).join(\'\')}</table>`;\n    $(\'.editing label\').firstChild.textContent=`位置${j+1}「${tokens[j]}」のV 第1成分 `;\n    $(\'.status\').textContent=`Python再計算 ${data.revision}回 ／ 更新した位置：${i+1} ${tokens[i]} ／ 計算結果を受信済み`;\n    showFlow();\n  }\n  function update(){const res=model.get(\'result\');if(!res||!res.tokens)return;clearTimeout(timeout);stop();busy=false;step=-1;data=res;$$(\'.app button,.app input,.app select\').forEach(e=>e.disabled=false);draw();}\n  $(\'select\').onchange=e=>send({preset:e.target.value,query:e.target.value===\'sentence\'?6:0,key:0,q:undefined,v:undefined,temperature:1,causal:false});\n  $(\'[aria-label="因果マスク"]\').onchange=e=>send({causal:e.target.checked});\n  $(\'[aria-label="温度 T"]\').onchange=e=>{if(e.target.checkValidity())send({temperature:+e.target.value});else e.target.reportValidity();};\n  const qchange=()=>{const a=$(\'[aria-label="Query 第1成分"]\'),b=$(\'[aria-label="Query 第2成分"]\');if(a.checkValidity()&&b.checkValidity())send({q:[+a.value,+b.value]});};\n  $(\'[aria-label="Query 第1成分"]\').onchange=qchange;$(\'[aria-label="Query 第2成分"]\').onchange=qchange;\n  $(\'[aria-label="Value 第1成分"]\').onchange=e=>{if(!e.target.checkValidity())return;const v=data.v.map(x=>[...x]);v[data.key][0]=+e.target.value;send({v});};\n  $(\'[data-action=reset]\').onclick=()=>send({q:undefined,v:undefined,query:data.preset===\'sentence\'?6:0,key:0,temperature:1,causal:false});\n  $(\'[data-action=calculate]\').onclick=qchange;\n  $$(\'[data-stage]\').forEach(b=>b.onclick=()=>{stop();stage=+b.dataset.stage;step=-1;draw();});\n  function advance(){stage=2;step=(step+1)%data.tokens.length;draw();}\n  $(\'[data-action=step]\').onclick=()=>{stop();advance();};\n  $(\'[data-action=play]\').onclick=()=>{stop();step=-1;advance();if(!matchMedia(\'(prefers-reduced-motion: reduce)\').matches)timer=setInterval(()=>{if(step===data.tokens.length-1){stop();return;}advance();},1150);};\n  $(\'[data-action=stop]\').onclick=stop;\n  model.on(\'change:result\',update);\n  const error=()=>{clearTimeout(timeout);busy=false;$$(\'.app button,.app input,.app select\').forEach(e=>e.disabled=false);$(\'.status\').textContent=model.get(\'error\');};model.on(\'change:error\',error);\n  update();return()=>{stop();clearTimeout(timeout);model.off(\'change:result\',update);model.off(\'change:error\',error);root.remove();};\n}\nexport default {render};\n'

import anywidget
import traitlets
from IPython.display import display

class AttentionLab(anywidget.AnyWidget):
    _esm = FRONTEND
    request = traitlets.Dict().tag(sync=True)
    result = traitlets.Dict().tag(sync=True)
    error = traitlets.Unicode("").tag(sync=True)

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._revision = 0
        self.observe(self._recalculate, names="request")
        self._recalculate({"new": {}})

    def _recalculate(self, change):
        try:
            answer = calculate(change["new"])
            self._revision += 1
            answer["revision"] = self._revision
            self.error = ""
            self.result = answer
        except (ValueError, TypeError, IndexError) as exc:
            self.error = "入力を確認してください：" + str(exc)

# 再実行時に古い表示・通信を閉じる。
if "attention_lab" in globals():
    attention_lab.close()
attention_lab = AttentionLab()
display(attention_lab)


## 2　模型から、学習済みのESM-2へ
ここからは実際に学習された小型モデル `esm2_t6_8M_UR50D`（6層、表現320次元）を使う。上の模型のQ・K・Vを渡すのではない。下のセルがモデルを読み込み、CPUで予測する。

取得できない場合は保存済みの8条件を使う。別のHTMLを開く必要はない。モデル準備が長く止まる場合はそのセルを中断し、次の表示セルへ進む。


In [ ]:
# @title 4. ESM-2を準備する（CPU）
import importlib.util, subprocess, sys, time, random
model = alphabet = converter = None
import socket
_previous_timeout = socket.getdefaulttimeout()
socket.setdefaulttimeout(30)
try:
    if importlib.util.find_spec("esm") is None:
        subprocess.run([sys.executable,"-m","pip","-q","install","fair-esm==2.0.0"],check=True,timeout=120)
    import torch, esm
    torch.set_num_threads(2)
    model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    model.eval()
    converter = alphabet.get_batch_converter()
    print("実モデルの準備完了：ESM-2 8M / CPU")
except Exception as exc:
    model = None
    print("実モデルの準備ができなかったため、計算済み出力を使います。",type(exc).__name__)

finally:
    socket.setdefaulttimeout(_previous_timeout)


## 3　同じ位置で、周囲の順序だけを変える
残基番号のボタンを押すと、Pythonが元の配列と並べ替えた配列の2条件を計算し、候補と確率を並べて表示する。最初は23番、続いて12・44・66番を見る。

**固定するもの**：配列長、アミノ酸組成、対象位置、隠した元の文字、モデル。**変えるもの**：周囲のアミノ酸の順序。位置は1始まりの残基番号である。入力では対象位置を［MASK］に置き換えるため、元の文字をモデルへ見せていない。

元の文字が常に1位になるとは限らず、元の配列が常に鋭い確率分布を示すとも限らない。上位5候補の確率は全語彙に対するsoftmaxの値で、5候補だけでは再正規化しない。


In [ ]:
# @title 5. ESM-2の予測と文脈の違いを操作する
CACHE = {'model': 'esm2_t6_8M_UR50D', 'fair_esm': '2.0.0', 'torch': '2.14.1', 'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG', 'device': 'cpu', 'threads': 4, 'cases': [{'position': 12, 'context': 'native', 'original': 'T', 'input_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG', 'top5': [{'aa': 'E', 'prob': 0.21892209351062775}, {'aa': 'T', 'prob': 0.1311466544866562}, {'aa': 'D', 'prob': 0.09625725448131561}, {'aa': 'K', 'prob': 0.09461205452680588}, {'aa': 'S', 'prob': 0.08928221464157104}], 'seconds': 0.010643959045410156}, {'position': 12, 'context': 'shuffled_context', 'original': 'T', 'input_sequence': 'SFELQNMEIVITYGQGKLNGQILEPDKLTQAIIGGIRQSTDIRLKQTLDEREHPRFKPEVSALKVKTLKTVGTDDL', 'top5': [{'aa': 'E', 'prob': 0.08507221937179565}, {'aa': 'D', 'prob': 0.0823030024766922}, {'aa': 'S', 'prob': 0.07692793756723404}, {'aa': 'G', 'prob': 0.0756354108452797}, {'aa': 'A', 'prob': 0.0690060406923294}], 'seconds': 0.0069828033447265625}, {'position': 23, 'context': 'native', 'original': 'I', 'input_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG', 'top5': [{'aa': 'L', 'prob': 0.32290321588516235}, {'aa': 'V', 'prob': 0.23876366019248962}, {'aa': 'I', 'prob': 0.23713551461696625}, {'aa': 'F', 'prob': 0.078935906291008}, {'aa': 'W', 'prob': 0.027193771675229073}], 'seconds': 0.006387948989868164}, {'position': 23, 'context': 'shuffled_context', 'original': 'I', 'input_sequence': 'SFELQNMVTVIYGQGKTNGQILIEEDKLDQAITGGIRQPTDIRLKQTLDERLHPRFKPEESALKVKTLKIVGTDSL', 'top5': [{'aa': 'L', 'prob': 0.10170266032218933}, {'aa': 'S', 'prob': 0.0834222361445427}, {'aa': 'I', 'prob': 0.08260525017976761}, {'aa': 'T', 'prob': 0.0759192481637001}, {'aa': 'A', 'prob': 0.07555261999368668}], 'seconds': 0.004967927932739258}, {'position': 44, 'context': 'native', 'original': 'I', 'input_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG', 'top5': [{'aa': 'L', 'prob': 0.21722222864627838}, {'aa': 'I', 'prob': 0.21533198654651642}, {'aa': 'V', 'prob': 0.2044026404619217}, {'aa': 'F', 'prob': 0.11902078986167908}, {'aa': 'Y', 'prob': 0.04333817586302757}], 'seconds': 0.006058931350708008}, {'position': 44, 'context': 'shuffled_context', 'original': 'I', 'input_sequence': 'SFKLDEMVTNIYGQEKTNGQKLIEQDLDIALTGGGRQPTDIRLIVQTLPEQLHPRFKIEESKLKVATLKIVGTDSR', 'top5': [{'aa': 'L', 'prob': 0.23462046682834625}, {'aa': 'A', 'prob': 0.13104519248008728}, {'aa': 'I', 'prob': 0.11886722594499588}, {'aa': 'F', 'prob': 0.10360973328351974}, {'aa': 'V', 'prob': 0.08632867783308029}], 'seconds': 0.005690813064575195}, {'position': 66, 'context': 'native', 'original': 'T', 'input_sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG', 'top5': [{'aa': 'V', 'prob': 0.19622531533241272}, {'aa': 'E', 'prob': 0.14044418931007385}, {'aa': 'I', 'prob': 0.1262766420841217}, {'aa': 'T', 'prob': 0.11564450711011887}, {'aa': 'K', 'prob': 0.09203518182039261}], 'seconds': 0.006050825119018555}, {'position': 66, 'context': 'shuffled_context', 'original': 'T', 'input_sequence': 'LIKLDEMVTNNDGQEKTYAQKQIEQDLDIFLTGDGRIPTSIRLVKRTPLQLHPGFQIKEEKLKVATSLGIVGTESR', 'top5': [{'aa': 'K', 'prob': 0.13401585817337036}, {'aa': 'E', 'prob': 0.12382804602384567}, {'aa': 'R', 'prob': 0.08806239068508148}, {'aa': 'L', 'prob': 0.08006556332111359}, {'aa': 'Q', 'prob': 0.06521821022033691}], 'seconds': 0.005728006362915039}]}
ESM_FRONTEND = 'function render({model, el}) {\n  const root=document.createElement(\'div\'); el.appendChild(root);\n  const sh=root.attachShadow({mode:\'open\'});\n  sh.innerHTML=`<style>\n:host{display:block;color:#172b42;background:white;font:16px/1.6 system-ui,sans-serif}*{box-sizing:border-box}.app{max-width:1100px;margin:auto;padding:20px}h2{font-size:24px;color:#174d91}h3{font-size:19px;margin:12px 0}p{margin:10px 0}.controls{display:flex;align-items:center;flex-wrap:wrap;gap:10px;margin:18px 0}button,select{font:inherit;background:white;color:#172b42;border:1px solid #9eb3c8;border-radius:5px;padding:8px 14px;cursor:pointer}button[aria-pressed=true]{background:#174d91;color:white}button:focus-visible,select:focus-visible{outline:3px solid #d57913}button:disabled,select:disabled{opacity:.55;cursor:wait}.status{padding:12px;background:#edf3fa;border-left:4px solid #174d91}.cards{display:grid;grid-template-columns:1fr 1fr;gap:24px;margin:18px 0}.panel{border-top:3px solid #174d91;padding-top:10px}.seq{display:flex;flex-wrap:wrap;gap:3px;margin:10px 0}.res{display:flex;flex-direction:column;align-items:center;width:26px;border:1px solid #d7e1eb;border-radius:3px;background:#f4f7fb;font-family:monospace;font-size:17px}.res small{font:10px system-ui;color:#526779}.res.changed{border-bottom:3px solid #b98b50}.res.mask{width:64px;background:#fff1dc;border:2px solid #c47713}.bars{width:100%;border-collapse:collapse}.bars th,.bars td{text-align:left;padding:7px 5px;border-bottom:1px solid #e0e7ef}.track{width:100%;height:20px;background:#edf2f7}.bar{height:100%;background:#2d669e;transition:width .3s}.prob{width:80px;font-variant-numeric:tabular-nums}.note{font-size:14px;color:#40576d}.error{color:#9f2020}.original{font-weight:bold}.scale{font-size:12px;display:flex;justify-content:space-between;margin:6px 5px 0 45px}@media(max-width:760px){.cards{grid-template-columns:1fr}.app{padding:12px}}@media(prefers-reduced-motion:reduce){.bar{transition:none}}\n</style><div class="app"><h2>ESM-2：隠したアミノ酸を周囲から予測する</h2>\n<p>ユビキチン（76残基）の同じ位置を隠し、元の配列と、周囲だけを並べ替えた配列を比較する。</p>\n<div class="controls"><strong>隠す残基番号</strong>${[12,23,44,66].map(p=>`<button data-position="${p}" aria-label="残基 ${p}" aria-pressed="false">${p}番</button>`).join(\'\')}</div>\n<div class="controls"><label>計算方法 <select aria-label="ESM-2の計算方法"><option value="live">ESM-2で計算する（CPU）</option><option value="saved">保存済みの実計算を表示する</option></select></label><button data-action="run">同じ条件で再計算</button></div>\n<div class="status" role="status" aria-live="polite"></div><div class="cards"></div>\n<p class="note">両条件で、配列長・アミノ酸組成・対象位置・隠した元の文字をそろえている。茶色の下線は元の配列と文字が異なる位置。並べ替えた配列は実験済みのタンパク質ではない。</p>\n<p class="note">棒の横軸は両方とも0〜100%。確率はモデルの全語彙に対するsoftmaxで、上位5候補だけでは正規化し直していない。これはアテンションの重みでも、機能や安定性の確率でもない。</p>\n<p class="note">I（Ile）・L（Leu）・V（Val）は非極性、K（Lys）は通常正に荷電、T（Thr）は極性の側鎖を持つ。元の文字が必ず1位になるわけではない。</p></div>`;\n  const $=s=>sh.querySelector(s), $$=s=>[...sh.querySelectorAll(s)];\n  const escape=s=>String(s).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\n  let position=23, requestId=0, timer=null;\n  function enabled(v){$$(\'button,select\').forEach(e=>e.disabled=!v);}\n  function send(){enabled(false);$(\'.cards\').replaceChildren();$(\'.status\').textContent=\'ColabのPythonで2条件を処理中…\';model.set(\'request\',{position,mode:$(\'select\').value,request_id:++requestId});model.save_changes();clearTimeout(timer);timer=setTimeout(()=>{enabled(true);$(\'.status\').textContent=\'Pythonの応答を待っています。Colabの接続を確かめてから「同じ条件で再計算」を押してください。\';},45000);}\n  function update(){const d=model.get(\'result\');if(!d||!d.rows)return;clearTimeout(timer);enabled(true);position=d.position;$(\'select\').value=d.mode;$$(\'[data-position]\').forEach(b=>b.setAttribute(\'aria-pressed\',String(+b.dataset.position===position)));$(\'.status\').classList.remove(\'error\');$(\'.status\').textContent=`${d.source} ／ Python更新 ${d.revision}回 ／ ${position}番：元の文字 ${d.original} ／ 計算結果を受信済み`;\n    $(\'.cards\').innerHTML=d.rows.map((r,i)=>`<section class="panel"><h3>${i===0?\'元の配列\':\'周囲の順序だけを変えた配列\'}</h3><div class="seq" aria-label="${i===0?\'元の配列\':\'並べ替えた配列\'}の入力">${[...r.input_sequence].map((a,k)=>`<span class="res ${k===position-1?\'mask\':a!==d.native_sequence[k]?\'changed\':\'\'}"><small>${k+1}</small>${k===position-1?\'[MASK]\':escape(a)}</span>`).join(\'\')}</div><p class="original">隠した元の文字：${escape(r.original)}</p><div class="scale"><span>0%</span><span>100%</span></div><table class="bars" aria-label="${i===0?\'元の配列\':\'並べ替えた配列\'}の上位5候補"><thead><tr><th>候補</th><th>予測確率</th><th class="prob">値</th></tr></thead><tbody>${r.top5.map(t=>`<tr><th>${escape(t.aa)}</th><td><div class="track"><div class="bar" style="width:${100*t.prob}%"></div></div></td><td class="prob">${(100*t.prob).toFixed(2)}%</td></tr>`).join(\'\')}</tbody></table></section>`).join(\'\');\n  }\n  function error(){const e=model.get(\'error\');if(!e)return;clearTimeout(timer);enabled(true);$(\'.cards\').replaceChildren();$(\'.status\').classList.add(\'error\');$(\'.status\').textContent=e;}\n  $$(\'[data-position]\').forEach(b=>b.onclick=()=>{position=+b.dataset.position;send();});$(\'select\').onchange=send;$(\'[data-action=run]\').onclick=send;\n  model.on(\'change:result\',update);model.on(\'change:error\',error);update();return()=>{clearTimeout(timer);model.off(\'change:result\',update);model.off(\'change:error\',error);root.remove();};\n}\nexport default {render};\n'
import random
import anywidget
import traitlets
from IPython.display import display

SEQUENCE = CACHE['sequence']
POSITIONS = (12, 23, 44, 66)
def make_sequence(position, context):
    if position not in POSITIONS or context not in ('native', 'shuffled_context'):
        raise ValueError('残基番号または比較条件が不正です。')
    if context == 'native':
        return SEQUENCE
    others = [a for i, a in enumerate(SEQUENCE, 1) if i != position]
    random.Random(7).shuffle(others)
    it = iter(others)
    return ''.join(a if i == position else next(it) for i, a in enumerate(SEQUENCE, 1))

def predict_esm(position, context, saved=False):
    sequence = make_sequence(position, context)
    if saved:
        return next(r for r in CACHE['cases'] if r['position'] == position and r['context'] == context)
    if model is None:
        raise RuntimeError('ESM-2の準備ができていません。「保存済みの実計算を表示する」を選ぶか、モデル準備セルを再実行してください。')
    _, _, tokens = converter([('ubiquitin', sequence)])
    tokens[0, position] = alphabet.mask_idx
    with torch.inference_mode():
        probabilities = model(tokens)['logits'][0, position].softmax(-1)
    values, indices = probabilities.topk(5)
    return dict(position=position, context=context, original=sequence[position-1], input_sequence=sequence,
                top5=[dict(aa=alphabet.get_tok(int(i)), prob=float(v)) for v, i in zip(values, indices)])

def compare_esm(request):
    position = int(request.get('position', 23))
    mode = request.get('mode', 'live' if model is not None else 'saved')
    if mode not in ('live', 'saved'):
        raise ValueError('計算方法が不正です。')
    rows = [predict_esm(position, c, mode == 'saved') for c in ('native', 'shuffled_context')]
    assert rows[0]['original'] == rows[1]['original']
    assert sorted(rows[0]['input_sequence']) == sorted(rows[1]['input_sequence'])
    return dict(position=position, mode=mode, rows=rows, original=rows[0]['original'], native_sequence=SEQUENCE,
                source='今回実行した予測（ESM-2 8M／CPU）' if mode == 'live' else '保存済みの実計算（ESM-2 8M／CPU）',
                request_id=request.get('request_id', 0))

class ESMLab(anywidget.AnyWidget):
    _esm = ESM_FRONTEND
    request = traitlets.Dict().tag(sync=True)
    result = traitlets.Dict().tag(sync=True)
    error = traitlets.Unicode('').tag(sync=True)

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._revision = 0
        self.observe(self._calculate, names='request')
        self._calculate({'new': {}})

    def _calculate(self, change):
        self.error = ''
        try:
            result = compare_esm(change['new'])
            self._revision += 1
            result['revision'] = self._revision
            self.result = result
        except Exception as exc:
            self.error = '計算できませんでした：' + str(exc)

if 'esm_lab' in globals():
    esm_lab.close()
esm_lab = ESMLab()
display(esm_lab)


## 観察からわかること
- 模型では、QとKの内積から参照の重みを作り、同じ重みでValueの各成分を足し合わせた。
- ESM-2では、周囲の順序を変えると予測候補や確率が変わる。対象の文字と組成をそろえた比較から、モデルが順序を含む文脈に依存することがわかる。
- ESM-2の予測確率、アテンションの重み、コンタクト確率は異なる量である。
- 高い予測確率だけで機能や安定性は確定できない。配列が学習データに含まれたかも、この演習では調べていない。
- 一つのタンパク質と小型モデルによる観察だけでは、スケーリング則や未知配列への汎化性能は評価できない。

[ESM公式実装](https://github.com/facebookresearch/esm)（fair-esm 2.0.0）／[Lin et al., Science (2023)](https://doi.org/10.1126/science.ade2574)／[PDB 1UBQ](https://www.rcsb.org/structure/1UBQ)／[anywidget](https://anywidget.dev/)
